# Notebook 13 — Confidence-Gated TrOCR + Lexicon

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import time
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE}")

Using device: mps


In [2]:
@dataclass
class Config:
    data_root: Path = Path("../data/pharmacy_lk")
    train_csv: str = "splits/train.csv"; val_csv: str = "splits/val.csv"; test_csv: str = "splits/test.csv"
    img_dir: str = "images"; img_col: str = "image_filename"; label_col: str = "medicine_name"
    processor_name: str = "microsoft/trocr-small-handwritten"
    trocr_ckpt: Path = Path("../checkpoints/benchmark_trocr/best")
    batch_size: int = 8; max_target_len: int = 24; num_workers: int = 0
    formulary_path: "Path | None" = None
    out_dir: Path = Path("../checkpoints/trocr_confgated")

CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

## 1. Load TrOCR, regenerate predictions WITH confidence
Confidence = exp(mean log-prob of generated tokens), from generate(..., output_scores=True).

In [3]:
processor = TrOCRProcessor.from_pretrained(CFG.processor_name)
model = VisionEncoderDecoderModel.from_pretrained(CFG.trocr_ckpt).to(DEVICE).eval()
print("fine-tuned TrOCR loaded")

class TrOCRDataset(Dataset):
    def __init__(self, csv_path, img_dir, cfg, processor):
        self.df = pd.read_csv(csv_path).dropna(subset=[cfg.label_col]).reset_index(drop=True)
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip()
        self.img_dir = Path(img_dir); self.cfg = cfg; self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / str(row[self.cfg.img_col])).convert("RGB")
        pv = self.processor(img, return_tensors="pt").pixel_values.squeeze(0)
        return {"pixel_values": pv, "text": row[self.cfg.label_col].lower(),
                "fname": str(row[self.cfg.img_col])}

def collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "text": [b["text"] for b in batch], "fname": [b["fname"] for b in batch]}

@torch.no_grad()
def trocr_predict_conf(loader):
    preds, confs, refs, files = [], [], [], []
    for batch in loader:
        out = model.generate(batch["pixel_values"].to(DEVICE), max_new_tokens=CFG.max_target_len,
                             output_scores=True, return_dict_in_generate=True)
        seqs = out.sequences
        # transition scores -> per-token log-probs of the chosen tokens
        tscores = model.compute_transition_scores(seqs, out.scores, normalize_logits=True)
        for i in range(seqs.size(0)):
            text = processor.decode(seqs[i], skip_special_tokens=True).strip().lower()
            row = tscores[i]
            valid = row[torch.isfinite(row)]
            conf = float(valid.mean().exp()) if valid.numel() else 0.0   # mean-token prob
            preds.append(text); confs.append(conf)
        refs += batch["text"]; files += batch["fname"]
    return preds, confs, refs, files

val_ds = TrOCRDataset(CFG.data_root / CFG.val_csv, CFG.data_root / CFG.img_dir, CFG, processor)
test_ds = TrOCRDataset(CFG.data_root / CFG.test_csv, CFG.data_root / CFG.img_dir, CFG, processor)
val_dl = DataLoader(val_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)

t0 = time.time()
val_raw, val_conf, val_refs, val_files = trocr_predict_conf(val_dl)
test_raw, test_conf, test_refs, test_files = trocr_predict_conf(test_dl)
print(f"predictions+confidence in {(time.time()-t0)/60:.1f} min | mean conf (val)={np.mean(val_conf):.3f}")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

fine-tuned TrOCR loaded
predictions+confidence in 0.8 min | mean conf (val)=0.188


## 2. Lexicon + confidence-gated decode

In [4]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

def corpus_metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs); exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot / max(chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}

lexicon = sorted(set(pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col]
                     .astype(str).str.strip().str.lower()))
if CFG.formulary_path and Path(CFG.formulary_path).exists():
    lexicon = sorted(set(lexicon) | {l.strip().lower() for l in open(CFG.formulary_path) if l.strip()})
by_len = defaultdict(list)
for w in lexicon: by_len[len(w)].append(w)
print(f"lexicon size: {len(lexicon)}")

def nearest(word, gap=3):
    if not word: return None, 10**9
    if word in by_len.get(len(word), ()): return word, 0
    best, bd = None, 10**9
    for L in range(len(word)-gap, len(word)+gap+1):
        for e in by_len.get(L, ()):
            d = edit_distance(word, e)
            if d < bd: best, bd = e, d
            if bd == 1: return best, bd
    return best, bd

def gated_decode(raw, confs, tau, conf_gate):
    """Snap to lexicon only when NOT confident (c < conf_gate) AND match close (d<=tau)."""
    out = []
    for p, c in zip(raw, confs):
        e, d = nearest(p)
        norm = d / max(len(p), 1)
        if e is not None and c < conf_gate and norm <= tau:
            out.append(e)
        else:
            out.append(p)
    return out

lexicon size: 1210


## 3. Tune (tau, conf_gate) on validation

In [5]:
taus = [0.2, 0.3, 0.4, 0.5]
gates = [0.50, 0.70, 0.85, 0.95, 1.01]   # 1.01 ~= "always apply" (gate off) for reference
records = []
for tau in taus:
    for g in gates:
        dec = gated_decode(val_raw, val_conf, tau, g)
        m = corpus_metrics(dec, val_refs)
        records.append({"tau": tau, "conf_gate": g, "val_EM": m["ExactMatch"]})
grid = pd.DataFrame(records)
best = grid.loc[grid.val_EM.idxmax()]
best_tau, best_gate = float(best.tau), float(best.conf_gate)
print(grid.pivot(index="tau", columns="conf_gate", values="val_EM").round(4).to_string())
print(f"\nbest on validation: tau={best_tau}, conf_gate={best_gate}, val_EM={best.val_EM:.4f}")

conf_gate    0.50    0.70    0.85    0.95    1.01
tau                                              
0.2        0.6063  0.6063  0.6114  0.6139  0.6165
0.3        0.6101  0.6114  0.6165  0.6190  0.6203
0.4        0.6114  0.6139  0.6241  0.6266  0.6278
0.5        0.6127  0.6152  0.6253  0.6266  0.6278

best on validation: tau=0.4, conf_gate=1.01, val_EM=0.6278


## 4. Test: TrOCR vs TrOCR+lexicon(plain) vs TrOCR+lexicon(gated)

In [6]:
trocr_only = corpus_metrics(test_raw, test_refs)
plain = gated_decode(test_raw, test_conf, 0.4, 1.01)          # gate off = plain lexicon
gated = gated_decode(test_raw, test_conf, best_tau, best_gate)
m_plain = corpus_metrics(plain, test_refs)
m_gated = corpus_metrics(gated, test_refs)

print(f"TEST (n={trocr_only['n']}):")
print(f"  TrOCR                    : CER {trocr_only['CER']:.4f} | EM {trocr_only['ExactMatch']:.4f}")
print(f"  TrOCR + lexicon (plain)  : CER {m_plain['CER']:.4f} | EM {m_plain['ExactMatch']:.4f}")
print(f"  TrOCR + lexicon (gated)  : CER {m_gated['CER']:.4f} | EM {m_gated['ExactMatch']:.4f}")

test_meta = pd.read_csv(CFG.data_root / CFG.test_csv)
seen_map = dict(zip(test_meta[CFG.img_col].astype(str), test_meta["seen_in_train"]))
def breakdown(preds, tag):
    g = {"seen": ([], []), "unseen": ([], [])}
    for p, r, f in zip(preds, test_refs, test_files):
        k = "seen" if seen_map.get(f, False) else "unseen"
        g[k][0].append(p); g[k][1].append(r)
    print(f"{tag}:")
    for kk, (P, R) in g.items():
        if R:
            m = corpus_metrics(P, R)
            print(f"  {kk:6s} (n={m['n']:3d}): EM {m['ExactMatch']:.4f}")
breakdown(test_raw, "TrOCR")
breakdown(gated, "TrOCR+lexicon(gated)")

# safety: gated vs plain
def safety(dec):
    fix = broke = 0
    for raw, d, r in zip(test_raw, dec, test_refs):
        if raw != d:
            if raw != r and d == r: fix += 1
            elif raw == r and d != r: broke += 1
    return fix, broke
print(f"\nplain lexicon:  fixes={safety(plain)[0]}, breaks={safety(plain)[1]}")
print(f"gated lexicon:  fixes={safety(gated)[0]}, breaks={safety(gated)[1]}")

pd.DataFrame([{"model": "TrOCR", **trocr_only},
              {"model": "TrOCR+lex(plain)", **m_plain},
              {"model": "TrOCR+lex(gated)", **m_gated}]).to_csv(CFG.out_dir / "gated_results.csv", index=False)

TEST (n=791):
  TrOCR                    : CER 0.2159 | EM 0.5689
  TrOCR + lexicon (plain)  : CER 0.2199 | EM 0.6119
  TrOCR + lexicon (gated)  : CER 0.2199 | EM 0.6119
TrOCR:
  seen   (n=615): EM 0.7008
  unseen (n=176): EM 0.1080
TrOCR+lexicon(gated):
  seen   (n=615): EM 0.7789
  unseen (n=176): EM 0.0284

plain lexicon:  fixes=48, breaks=14
gated lexicon:  fixes=48, breaks=14


## 5. Thesis framing
- If gated > plain AND breaks drop: confidence gating recovers the unseen reads while keeping
  the seen-name gains — "a confidence-gated lexicon improves even a SOTA pretrained recogniser
  WITHOUT harming its out-of-vocabulary predictions." Clean, complete contribution.
- Report the (tau, conf_gate) grid and the fixes/breaks for plain vs gated as the evidence.
- This also closes the loop with M2+: confidence-aware decoding mattered little for the weak
  CRNN (it fails on unseen anyway) but matters for the strong TrOCR — a coherent story about
  WHEN confidence information is useful.